# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mashfiqmahi/assignment_FLyRank-AI/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Distributions. Impressions and clicks are heavily right-skewed: median impressions (731) is far below the mean (5,200), meaning a small number of high-traffic pages pull the average up while most pages sit much lower. Because of this, all group comparisons below use medians and weighted totals (total clicks ÷ total impressions), never a plain average of per-page rates. avg_position = 0 for 1,205 rows means "no position data," not rank zero, and is excluded before any position-based test.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("content_refresh_anonymized.csv")

key_fields = ["impressions_90d", "clicks_90d", "ctr", "avg_position",
              "days_since_last_update", "engagement_rate", "scroll_rate",
              "ai_traffic_pct", "trend_pct", "word_count"]

print("=== Basic stats on key fields ===")
print(df[key_fields].describe().T[["count", "mean", "50%", "std", "min", "max"]])

print("\n=== Mean vs Median check (big gap = heavy tail / outliers) ===")
for c in ["impressions_90d", "clicks_90d"]:
    print(f"{c}: mean={df[c].mean():.1f}  median={df[c].median()}  max={df[c].max()}")

print("\n=== avg_position: rows with 0 (means 'no data', not rank zero!) ===")
print((df["avg_position"] == 0).sum(), "of", len(df), "rows")

=== Basic stats on key fields ===
                          count         mean      50%           std    min  \
impressions_90d         30000.0  5200.366300   731.00  16838.019547    1.0   
clicks_90d              30000.0    16.097333     1.00     75.076958    0.0   
ctr                     30000.0     0.510733     0.07      3.279162    0.0   
avg_position            30000.0    16.342380    10.80     15.216790    0.0   
days_since_last_update  30000.0    46.098300    20.00     42.078709    1.0   
engagement_rate         30000.0     2.534520     0.00      8.310096    0.0   
scroll_rate             29875.0    18.212921     5.00     29.472768    0.0   
ai_traffic_pct          30000.0     0.768196     0.00      7.429454    0.0   
trend_pct               26612.0    -4.785969   -33.50    473.861780 -100.0   
word_count              22301.0  3107.760325  2877.00   1452.382598    8.0   

                             max  
impressions_90d         517715.0  
clicks_90d                4178.0  
ct

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

## Test 1

In [ ]:
d = df[df["impressions_90d"] > 0].copy()
d["update_bucket"] = pd.cut(
    d["days_since_last_update"], bins=[0, 30, 90, 180, 10000],
    labels=["0-30d", "31-90d", "91-180d", "181d+"]
)
g1 = d.groupby("update_bucket", observed=True).agg(
    n=("impressions_90d", "size"),
    median_impressions=("impressions_90d", "median"),
    total_impr=("impressions_90d", "sum"),
    total_clicks=("clicks_90d", "sum"),
)
g1["weighted_ctr_pct"] = (g1["total_clicks"] / g1["total_impr"] * 100).round(3)
print(g1)

                   n  median_impressions  total_impr  total_clicks  \
update_bucket                                                        
0-30d          20480               470.0    86008096        281137   
31-90d           175               510.0     1138681          1695   
91-180d         9171              1692.0    68660206        199623   
181d+            174                15.5      204006           465   

               weighted_ctr_pct  
update_bucket                    
0-30d                     0.327  
31-90d                    0.149  
91-180d                   0.291  
181d+                     0.228  


staleness vs. traffic:

Verdict: MIXED. Extremely stale pages (181+ days since update, n=174) show far lower median impressions (15.5) than the rest, but the pattern isn't smooth — pages updated 91-180 days ago actually out-traffic freshly-updated ones. Middle buckets are thin (n=175, n=174), and there's a likely selection effect: editors may update pages that already matter, so causation can't be claimed either way.

## Test 2

In [ ]:
d2 = df[df["position_tier"] != "no_data"].copy()
order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
g2 = d2.groupby("position_tier", observed=True).agg(
    n=("ctr", "size"),
    total_impr=("impressions_90d", "sum"),
    total_clicks=("clicks_90d", "sum"),
    median_impressions=("impressions_90d", "median"),
)
g2["weighted_ctr_pct"] = (g2["total_clicks"] / g2["total_impr"] * 100).round(3)
print(g2.reindex(order))

                   n  total_impr  total_clicks  median_impressions  \
position_tier                                                        
top_3           2321     7032960         34355                 3.0   
page_1         11814    89575437        313804              1179.5   
striking        7304    22992054         79754               874.5   
page_3_5        7242    35182261         54499               811.5   
deep            1319     1228277           508               218.0   

               weighted_ctr_pct  
position_tier                    
top_3                     0.488  
page_1                    0.350  
striking                  0.347  
page_3_5                  0.155  
deep                      0.041  


position tier vs. CTR:

Verdict: CONFIRMED. Weighted CTR falls cleanly from 0.488% (top_3) to 0.041% (deep) as ranking worsens, across healthy sample sizes (n=1,319 to n=11,814). Caveat: top_3's median impression volume is only 3, so its CTR advantage sits on very low-traffic pages, not FlyRank's biggest earners.

## Test 3

In [ ]:
d3 = df[df["word_count_tier"].notna() & (df["pageviews_90d"] > 0)].copy()
order3 = ["<1000", "1000-2000", "2000-3500", "3500+"]
g3 = d3.groupby("word_count_tier", observed=True).agg(
    n=("scroll_rate", "size"),
    median_scroll_rate=("scroll_rate", "median"),
    median_engagement_rate=("engagement_rate", "median"),
)
print(g3.reindex(order3))

                     n  median_scroll_rate  median_engagement_rate
word_count_tier                                                   
<1000              973              50.000                     0.0
1000-2000         3778              10.870                     0.0
2000-3500        11150               7.325                     0.0
3500+             6278               5.985                     0.0


word count vs. engagement:

Verdict:  OPPOSITE. Shorter articles (<1000 words) show far higher median scroll rate (50.0) than long ones (3500+ words: 6.0) — the reverse of "longer content = more engagement." engagement_rate was excluded from the headline finding because its median is 0 in every bucket (too zero-inflated to compare); scroll_rate carried the real signal.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

Flag-linked test — does the needs_ctr_fix assumption hold? The product flag assumes good-position pages with low CTR are worth reviewing. After applying a volume floor (≥100 impressions, to avoid noise from low-traffic pages), we observed that 40.7% of page_1 pages (n=8,633) and 56.8% of top_3 pages (n=533) click at less than half their tier's expected weighted CTR. The underlying assumption is observable in the data — good position clearly does not guarantee good CTR. However, this is a large share of pages, not a small outlier group, suggesting the flag alone may over-trigger for editorial review without further prioritization. This is a decision-support finding only: we have not tested whether "fixing" these pages actually raises CTR afterward, since that requires a before/after comparison, not a single snapshot.

In [ ]:
# Apply a volume floor first — CTR on 3 impressions is meaningless noise
d = df[(df["position_tier"].isin(["top_3", "page_1"])) &
       (df["impressions_90d"] >= 100)].copy()
print("Rows after volume floor (impressions_90d >= 100):", len(d))

# What's the "expected" (weighted) CTR for each tier, on this same slice?
tier_stats = d.groupby("position_tier").apply(
    lambda x: pd.Series({
        "n": len(x),
        "expected_weighted_ctr_pct": x["clicks_90d"].sum() / x["impressions_90d"].sum() * 100
    })
)
print(tier_stats)

# Flag a page as "underperforming" if its own CTR is under half of its tier's expected CTR
d = d.merge(tier_stats["expected_weighted_ctr_pct"], left_on="position_tier", right_index=True)
d["underperforming"] = d["ctr"] < (0.5 * d["expected_weighted_ctr_pct"])

print("\nShare of pages flagged as CTR-underperforming, per tier:")
print(d.groupby("position_tier")["underperforming"].mean().round(3))

Rows after volume floor (impressions_90d >= 100): 9166
                    n  expected_weighted_ctr_pct
position_tier                                   
page_1         8633.0                   0.349854
top_3           533.0                   0.487133

Share of pages flagged as CTR-underperforming, per tier:
position_tier
page_1    0.407
top_3     0.568
Name: underperforming, dtype: float64


/tmp/ipykernel_1316/24016323.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tier_stats = d.groupby("position_tier").apply(


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

* "stale content" alone is a weak signal for prioritizing pages

* ranking position is a genuinely reliable signal for click-through rate and is safe to trust in a scoring rule, as long as very-low-traffic "top 3" pages aren't mistaken for big wins;
* the existing CTR-fix assumption is real but far from rare —  so editors start with the highest-value fixes first rather than working through an unordered, oversized list.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.